In [11]:
"""
Concept Bottleneck Model
Classes: Melanoma vs Nevus
"""

import os
import random
import warnings
import numpy as np
import pandas as pd
from PIL import Image
import copy
import sys
import builtins

warnings.filterwarnings("ignore")

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
import torchvision.models as models
import torch.nn.functional as F

from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, f1_score
from sklearn.utils.class_weight import compute_class_weight
from sklearn.model_selection import train_test_split

# ============================================================
# CONFIG
# ============================================================

class Config:

    BATCH_SIZE    = 16
    EPOCHS        = 20
    IMG_SIZE      = 224
    VAL_SIZE      = 0.10

    LEARNING_RATE = 1e-4
    ETA_MIN       = 1e-6
    L1_LAMBDA     = 1e-3
    WEIGHT_DECAY  = 1e-3

    META_DIR   = "meta"
    IMAGES_DIR = "ISIC-images"
    OUTPUT_DIR = "models"

    # BACKBONE   = "densenet201"
    BACKBONE   = "efficientnet_b0"
    FREEZE_PCT = 0.0
    LAMBDA1    = 1.0
    LAMBDA2    = 1.0

    FOCAL_GAMMA_LABEL    = None
    FOCAL_GAMMA_CONCEPT  = None
    CONCEPT_GAMMA_VALUES = None

    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    CONCEPT_COLUMNS = [
        "reticular_lines",
        "eccentric_structureless_area",
        "peripheral_dots_globules",
        "grey_structures",
        "white_structures",
        "follicular_invasion",
        "vessel_pattern",
        "pseudopods_radial_lines",
        "acral_pattern",
        "scar_pigmentation",
    ]

    TARGET_COLUMN = "diagnosis"
    IMAGE_COLUMN  = "image_id"

    DATASET_NAME = "clean_concepts_dataset_symbolic.csv"

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False
    os.environ['PYTHONHASHSEED']       = str(seed)

set_seed(42)
config = Config()
os.makedirs(config.OUTPUT_DIR, exist_ok=True)

# ============================================================
# BACKBONE
# ============================================================

def get_backbone(name, freeze_pct=0.0):
    # ===== EFFICIENTNET =====
    if name == "efficientnet_b0":
        model = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.DEFAULT)
        features = model.features
        feature_dim = 1280
    elif name == "efficientnet_b1":
        model = models.efficientnet_b1(weights=models.EfficientNet_B1_Weights.DEFAULT)
        features = model.features
        feature_dim = 1280
    elif name == "efficientnet_b2":
        model = models.efficientnet_b2(weights=models.EfficientNet_B2_Weights.DEFAULT)
        features = model.features
        feature_dim = 1408
    elif name == "efficientnet_b3":
        model = models.efficientnet_b3(weights=models.EfficientNet_B3_Weights.DEFAULT)
        features = model.features
        feature_dim = 1536
    elif name == "efficientnet_b4":
        model = models.efficientnet_b4(weights=models.EfficientNet_B4_Weights.DEFAULT)
        features = model.features
        feature_dim = 1792
    elif name == "efficientnet_b5":
        model = models.efficientnet_b5(weights=models.EfficientNet_B5_Weights.DEFAULT)
        features = model.features
        feature_dim = 2048
    elif name == "efficientnet_b6":
        model = models.efficientnet_b6(weights=models.EfficientNet_B6_Weights.DEFAULT)
        features = model.features
        feature_dim = 2304
    elif name == "efficientnet_b7":
        model = models.efficientnet_b7(weights=models.EfficientNet_B7_Weights.DEFAULT)
        features = model.features
        feature_dim = 2560

    # ===== DENSENET =====
    elif name == "densenet121":
        model = models.densenet121(weights=models.DenseNet121_Weights.DEFAULT)
        features = model.features
        feature_dim = 1024
    elif name == "densenet161":
        model = models.densenet161(weights=models.DenseNet161_Weights.DEFAULT)
        features = model.features
        feature_dim = 2208
    elif name == "densenet169":
        model = models.densenet169(weights=models.DenseNet169_Weights.DEFAULT)
        features = model.features
        feature_dim = 1664
    elif name == "densenet201":
        model = models.densenet201(weights=models.DenseNet201_Weights.DEFAULT)
        features = model.features
        feature_dim = 1920

    # ===== RESNET =====
    elif name == "resnet18":
        model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
        features = nn.Sequential(*list(model.children())[:-2])
        feature_dim = 512
    elif name == "resnet34":
        model = models.resnet34(weights=models.ResNet34_Weights.DEFAULT)
        features = nn.Sequential(*list(model.children())[:-2])
        feature_dim = 512
    elif name == "resnet50":
        model = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
        features = nn.Sequential(*list(model.children())[:-2])
        feature_dim = 2048
    elif name == "resnet101":
        model = models.resnet101(weights=models.ResNet101_Weights.DEFAULT)
        features = nn.Sequential(*list(model.children())[:-2])
        feature_dim = 2048
    elif name == "resnet152":
        model = models.resnet152(weights=models.ResNet152_Weights.DEFAULT)
        features = nn.Sequential(*list(model.children())[:-2])
        feature_dim = 2048

    # ===== WIDE RESNET =====
    elif name == "wide_resnet50_2":
        model = models.wide_resnet50_2(weights=models.Wide_ResNet50_2_Weights.DEFAULT)
        features = nn.Sequential(*list(model.children())[:-2])
        feature_dim = 2048
    elif name == "wide_resnet101_2":
        model = models.wide_resnet101_2(weights=models.Wide_ResNet101_2_Weights.DEFAULT)
        features = nn.Sequential(*list(model.children())[:-2])
        feature_dim = 2048

    else:
        raise ValueError(
            f"Unknown backbone: {name}. "
            "Available: efficientnet_b0-b7, densenet121/161/169/201, "
            "resnet18/34/50/101/152, wide_resnet50_2/101_2"
        )

    # ===== FREEZE LAYERS =====
    total_layers = len(features)
    freeze_up_to = int(total_layers * freeze_pct)

    for i, layer in enumerate(features):
        if i < freeze_up_to:
            for param in layer.parameters():
                param.requires_grad = False
        else:
            for param in layer.parameters():
                param.requires_grad = True

    print(f"Froze first {freeze_up_to} out of {total_layers} layers in backbone ({freeze_pct*100:.0f}%)")

    return features, feature_dim


# ============================================================
# CONCEPT ENCODERS
# ============================================================

def build_global_concept_encoders(full_meta_df):
    concept_info, label_encoders = {}, {}
    for c in config.CONCEPT_COLUMNS:
        le = LabelEncoder()
        le.fit(full_meta_df[c].astype(str).fillna("unknown").values)
        label_encoders[c] = le
        concept_info[c]   = {"n_classes": len(le.classes_)}
    return concept_info, label_encoders


# ============================================================
# DATASET
# ============================================================

class DermDataset(Dataset):
    def __init__(self, df, images_dir, concept_info, label_encoders, transform=None):
        self.df             = df.reset_index(drop=True)
        self.images_dir     = images_dir
        self.transform      = transform
        self.concept_info   = concept_info
        self.label_encoders = label_encoders

        self.image_paths = {}
        for root, _, files in os.walk(images_dir):
            for f in files:
                if f.lower().endswith((".jpg", ".jpeg", ".png")):
                    full = os.path.join(root, f)
                    rel  = os.path.relpath(full, images_dir).replace("\\", "/").lower()
                    self.image_paths[rel] = full

        for c in config.CONCEPT_COLUMNS:
            le = self.label_encoders[c]
            self.df[c] = le.transform(df[c].astype(str).fillna("unknown").values)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row      = self.df.iloc[idx]
        rel_path = str(row[config.IMAGE_COLUMN]).replace("\\", "/").lower()
        if rel_path not in self.image_paths:
            raise FileNotFoundError(f"Image not found: {rel_path}")

        image = Image.open(self.image_paths[rel_path]).convert("RGB")
        if self.transform:
            image = self.transform(image)

        concepts = torch.tensor(
            [int(row[c]) for c in config.CONCEPT_COLUMNS], dtype=torch.long
        )
        label = torch.tensor(int(row[config.TARGET_COLUMN]), dtype=torch.long)
        return image, concepts, label


# ============================================================
# TRANSFORMS
# ============================================================

def get_train_transforms():
    return transforms.Compose([
        transforms.RandomResizedCrop(config.IMG_SIZE, scale=(0.80, 1.0)),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomVerticalFlip(p=0.5),
        transforms.RandomRotation(degrees=180),
        transforms.RandomApply([
            transforms.RandomAffine(degrees=0, translate=(0.05, 0.05))
        ], p=0.3),
        transforms.ColorJitter(
            brightness=0.2,
            contrast=0.2,
            saturation=0.15,
            hue=0.04,
        ),
        transforms.RandomApply([
            transforms.GaussianBlur(kernel_size=5, sigma=(0.1, 1.0))
        ], p=0.3),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    ])


def get_val_transforms():
    return transforms.Compose([
        transforms.Resize((config.IMG_SIZE, config.IMG_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    ])


# ============================================================
# MODEL
# ============================================================

class ConceptBottleneckModel(nn.Module):

    def __init__(self, backbone_name, concept_info, num_classes=2):
        super().__init__()

        self.feature_extractor, self.feature_dim = get_backbone(backbone_name, freeze_pct=config.FREEZE_PCT)
        self.pool         = nn.AdaptiveAvgPool2d(1)
        self.concept_info = concept_info

        self.concept_heads = nn.ModuleDict({
            c: nn.Linear(self.feature_dim, concept_info[c]["n_classes"])
            for c in config.CONCEPT_COLUMNS
        })

        total_concept_dim = sum(
            concept_info[c]["n_classes"] for c in config.CONCEPT_COLUMNS
        )

        self.classifier = nn.Sequential(
            nn.Linear(total_concept_dim, 64),
            nn.LayerNorm(64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, num_classes),
        )

    def forward(self, x):
        feats = self.feature_extractor(x)
        feats = self.pool(feats).view(feats.size(0), -1)

        concept_logits   = []
        one_hot_concepts = []

        for c in config.CONCEPT_COLUMNS:
            logits = self.concept_heads[c](feats)
            concept_logits.append(logits)

            hard = F.one_hot(
                torch.argmax(logits, dim=1),
                num_classes=self.concept_info[c]["n_classes"]
            ).float().detach()

            one_hot_concepts.append(hard)

        logits_y = self.classifier(torch.cat(one_hot_concepts, dim=1))
        return concept_logits, logits_y


# ============================================================
# LOSS FUNCTIONS
# ============================================================

class WeightedCrossEntropyLoss(nn.Module):
    def __init__(self, weight=None, reduction="mean"):
        super().__init__()
        self.weight    = weight
        self.reduction = reduction

    def forward(self, logits, targets):
        return F.cross_entropy(logits, targets, weight=self.weight,
                               reduction=self.reduction)


class FocalLoss(nn.Module):
    def __init__(self, alpha=None, gamma=2.0, reduction="mean"):
        super().__init__()
        self.alpha     = alpha
        self.gamma     = gamma
        self.reduction = reduction

    def forward(self, logits, targets):
        ce   = F.cross_entropy(logits, targets, weight=self.alpha, reduction="none")
        loss = ((1 - torch.exp(-ce)) ** self.gamma) * ce
        return loss.mean() if self.reduction == "mean" else loss.sum()


class CBMLoss(nn.Module):
    def __init__(self, class_weights_label, concept_class_weights):
        super().__init__()

        if config.FOCAL_GAMMA_LABEL is not None:
            self.label_loss      = FocalLoss(alpha=class_weights_label,
                                             gamma=config.FOCAL_GAMMA_LABEL)
            self.label_loss_type = f"Focal (gamma={config.FOCAL_GAMMA_LABEL})"
        else:
            self.label_loss      = WeightedCrossEntropyLoss(weight=class_weights_label)
            self.label_loss_type = "Weighted CrossEntropy"

        self.concept_losses     = nn.ModuleList()
        self.concept_loss_types = []

        for c in config.CONCEPT_COLUMNS:
            gamma = (config.CONCEPT_GAMMA_VALUES or {}).get(c, config.FOCAL_GAMMA_CONCEPT)
            if gamma is not None:
                loss = FocalLoss(alpha=concept_class_weights[c], gamma=gamma)
                self.concept_loss_types.append(f"Focal (gamma={gamma})")
            else:
                loss = WeightedCrossEntropyLoss(weight=concept_class_weights[c])
                self.concept_loss_types.append("Weighted CE")
            self.concept_losses.append(loss)

    def forward(self, concept_preds, concept_true, logits_y, labels):
        loss_c = sum(
            self.concept_losses[i](concept_preds[i], concept_true[:, i])
            for i in range(len(config.CONCEPT_COLUMNS))
        ) / len(config.CONCEPT_COLUMNS)

        return config.LAMBDA1 * loss_c + config.LAMBDA2 * self.label_loss(logits_y, labels)


# ============================================================
# TRAIN AND EVAL
# ============================================================

def train_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss = 0.0
    for imgs, concepts, labels in loader:
        imgs, concepts, labels = (imgs.to(config.DEVICE),
                                  concepts.to(config.DEVICE),
                                  labels.to(config.DEVICE))
        optimizer.zero_grad()
        concept_preds, logits_y = model(imgs)

        base_loss = criterion(concept_preds, concepts, logits_y, labels)

        first_layer_weight = model.classifier[0].weight
        l1_loss = config.L1_LAMBDA * torch.norm(first_layer_weight, p=1)

        loss = base_loss + l1_loss

        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)


def evaluate(model, loader):
    model.eval()
    all_preds         = []
    all_labels        = []
    concept_preds_all = [[] for _ in config.CONCEPT_COLUMNS]
    concept_true_all  = [[] for _ in config.CONCEPT_COLUMNS]

    with torch.no_grad():
        for imgs, concepts, labels in loader:
            imgs, concepts, labels = (imgs.to(config.DEVICE),
                                      concepts.to(config.DEVICE),
                                      labels.to(config.DEVICE))
            concept_logits, logits_y = model(imgs)
            all_preds.extend(logits_y.argmax(dim=1).cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            for i, logits in enumerate(concept_logits):
                concept_preds_all[i].extend(logits.argmax(dim=1).cpu().numpy())
                concept_true_all[i].extend(concepts[:, i].cpu().numpy())

    acc   = accuracy_score(all_labels, all_preds)
    f1    = f1_score(all_labels, all_preds, average="macro")
    c_acc = np.mean([accuracy_score(concept_true_all[i], concept_preds_all[i])
                     for i in range(len(config.CONCEPT_COLUMNS))])
    c_f1  = np.mean([f1_score(concept_true_all[i], concept_preds_all[i],
                               average="macro", zero_division=0)
                     for i in range(len(config.CONCEPT_COLUMNS))])
    return acc, f1, c_acc, c_f1


# ============================================================
# BUILD AND TRAIN
# ============================================================

def build_model():

    # ── Metadata ─────────────────────────────────────────────────────────
    meta = pd.read_csv(
        os.path.join(config.META_DIR, config.DATASET_NAME)
    )
    meta[config.TARGET_COLUMN] = (
        meta[config.TARGET_COLUMN].astype(str).str.lower()
        .apply(lambda x: 1 if "melanoma" in x else 0)
    )
    concept_info, label_encoders = build_global_concept_encoders(meta)

    # ── Splits ───────────────────────────────────────────────────────────
    full_train_df = meta[meta['split'] == 'train'].copy()
    test_df       = meta[meta['split'] == 'test'].copy()

    train_df, valid_df = train_test_split(
        full_train_df,
        test_size=config.VAL_SIZE,
        random_state=42,
        stratify=full_train_df[config.TARGET_COLUMN],
    )

    print("Dataset splits:")
    print(f"  Train:      {len(train_df):,} samples")
    print(f"  Validation: {len(valid_df):,} samples  ({int(config.VAL_SIZE*100)}%)")
    print(f"  Test:       {len(test_df):,} samples")

    # ── Class weights ────────────────────────────────────────────────────
    y_train = train_df[config.TARGET_COLUMN].values
    present = np.unique(y_train)
    wp      = compute_class_weight("balanced", classes=present, y=y_train)
    fw      = np.ones(2)
    for cls, w in zip(present, wp):
        fw[cls] = w
    label_weights = torch.tensor(fw, dtype=torch.float32).to(config.DEVICE)

    concept_class_weights = {}
    for c in config.CONCEPT_COLUMNS:
        le        = label_encoders[c]
        y_enc     = le.transform(train_df[c].astype(str).fillna("unknown"))
        present_c = np.unique(y_enc)
        wp_c      = compute_class_weight("balanced", classes=present_c, y=y_enc)
        fw_c      = np.ones(len(le.classes_))
        for cls, w in zip(present_c, wp_c):
            fw_c[cls] = w
        concept_class_weights[c] = torch.tensor(fw_c, dtype=torch.float32).to(config.DEVICE)

    # ── Datasets / loaders ───────────────────────────────────────────────
    mk = lambda df, t: DermDataset(df, config.IMAGES_DIR, concept_info,
                                   label_encoders, t)
    train_dataset = mk(train_df, get_train_transforms())
    valid_dataset = mk(valid_df, get_val_transforms())
    test_dataset  = mk(test_df,  get_val_transforms())

    train_loader = DataLoader(train_dataset, batch_size=config.BATCH_SIZE,
                              shuffle=True, num_workers=2, pin_memory=True)
    valid_loader = DataLoader(valid_dataset, batch_size=config.BATCH_SIZE,
                              num_workers=2, pin_memory=True)
    test_loader  = DataLoader(test_dataset,  batch_size=config.BATCH_SIZE,
                              num_workers=2, pin_memory=True)

    # ── Model ────────────────────────────────────────────────────────────
    print(f"\nUsing backbone: {config.BACKBONE} ({int(config.FREEZE_PCT*100)}% of layers frozen)")
    print(f"L1 regularization on first classifier layer: lambda = {config.L1_LAMBDA}")
    print(f"Weight decay on backbone: {config.WEIGHT_DECAY}")
    model = ConceptBottleneckModel(config.BACKBONE, concept_info).to(config.DEVICE)
    total_params     = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    frozen_params    = total_params - trainable_params
    print(f"Total parameters:     {total_params:,}")
    print(f"Trainable parameters: {trainable_params:,}")
    print(f"Frozen parameters:    {frozen_params:,}")

    # ── Loss ─────────────────────────────────────────────────────────────
    criterion = CBMLoss(label_weights, concept_class_weights)
    # print(f"\nLoss Configuration:")
    # print(f"  Label Loss: {criterion.label_loss_type}")
    # print(f"  L1 Lambda: {config.L1_LAMBDA} (applied to first classifier layer only)")
    # for i, c in enumerate(config.CONCEPT_COLUMNS):
    #     print(f"  Concept '{c}': {criterion.concept_loss_types[i]}")

    # ── Optimizer + scheduler ─────────────────────────────────────────────
    backbone_params     = list(model.feature_extractor.parameters())
    non_backbone_params = (
        list(model.concept_heads.parameters()) +
        list(model.classifier.parameters())
    )
    optimizer = optim.Adam([
        {"params": backbone_params,     "lr": config.LEARNING_RATE, "weight_decay": config.WEIGHT_DECAY},
        {"params": non_backbone_params, "lr": config.LEARNING_RATE, "weight_decay": 0.0},
    ])
    scheduler = optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=config.EPOCHS, eta_min=config.ETA_MIN
    )

    # print(f"\nScheduler: CosineAnnealingLR  "
    #       f"(T_max={config.EPOCHS}, eta_min={config.ETA_MIN})")
    # print(f"Training for {config.EPOCHS} epochs\n")

    # ── Training loop ─────────────────────────────────────────────────────
    best_f1, best_state, best_epoch = -1, copy.deepcopy(model.state_dict()), 0

    for epoch in range(config.EPOCHS):
        train_loss                      = train_epoch(model, train_loader, optimizer, criterion)
        val_acc, val_f1, vc_acc, vc_f1 = evaluate(model, valid_loader)
        scheduler.step()
        lr = scheduler.get_last_lr()[0]

        is_best = val_f1 > best_f1
        if is_best:
            best_f1    = val_f1
            best_state = copy.deepcopy(model.state_dict())
            best_epoch = epoch + 1

        print(f"Epoch {epoch+1:>3}/{config.EPOCHS} | "
              f"LR: {lr:.2e} | "
              f"Train Loss: {train_loss:.4f} | "
              f"Concept Acc: {vc_acc:.4f} | "
              f"Concept F1: {vc_f1:.4f} | "
              f"Label Acc: {val_acc:.4f} | "
              f"Label F1: {val_f1:.4f}"
              + (" ← best" if is_best else ""))

    # ── Test ─────────────────────────────────────────────────────────────
    model.load_state_dict(best_state)
    test_acc, test_f1, test_c_acc, test_c_f1 = evaluate(model, test_loader)

    print(f"\n{'='*60}")
    print(f"FINAL TEST RESULTS — {config.BACKBONE} ({int(config.FREEZE_PCT*100)}% frozen)")
    print(f"Best checkpoint: epoch {best_epoch}  (val F1 = {best_f1:.4f})")
    print(f"{'='*60}")
    print(f"Label Loss:       {criterion.label_loss_type}")
    print(f"L1 Lambda:        {config.L1_LAMBDA} (first classifier layer only)")
    print(f"Weight Decay:     {config.WEIGHT_DECAY} (backbone only)")
    print(f"Label Accuracy:   {test_acc:.4f}")
    print(f"Label F1 Score:   {test_f1:.4f}")
    print(f"Concept Accuracy: {test_c_acc:.4f}")
    print(f"Concept F1 Score: {test_c_f1:.4f}")
    print(f"{'='*60}")

    # ── Save ─────────────────────────────────────────────────────────────
    torch.save(
        {
            "model_state_dict":    model.state_dict(),
            "concept_info":        concept_info,
            "label_encoders":      label_encoders,
            "backbone":            config.BACKBONE,
            "freeze_pct":          config.FREEZE_PCT,
            "weight_decay":        config.WEIGHT_DECAY,
            "l1_lambda":           config.L1_LAMBDA,
            "label_loss_config":   criterion.label_loss_type,
            "concept_loss_config": criterion.concept_loss_types,
            "best_epoch":          best_epoch,
            "best_val_f1":         best_f1,
            "test_results": {
                "accuracy":         test_acc,
                "f1":               test_f1,
                "concept_accuracy": test_c_acc,
                "concept_f1":       test_c_f1,
            },
        },
        os.path.join(config.OUTPUT_DIR, f"cbm_model_{config.BACKBONE}.pth"),
    )

In [12]:
# if __name__ == "__main__":
#     build_model()

# Explanation module

In [13]:
# ====================================================================
#   SOFI-BASED EXPLAINER FOR CONCEPT RANKING
# ====================================================================

import os
import numpy as np
import torch
import torch.nn as nn
import pandas as pd
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader
from sklearn.preprocessing import LabelEncoder
from torch.serialization import add_safe_globals
from openai import OpenAI
import torch.nn.functional as F
from collections import OrderedDict

# backbone_model = "cbm_model_densenet201.pth"
backbone_model = "cbm_model_efficientnet_b0.pth"

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ============================================================
# INITIALIZE OPENAI CLIENT
# ============================================================
def load_api_key(file_path="chatgpt_api_key.txt"):
    if not os.path.exists(file_path):
        raise FileNotFoundError(f"API key file not found: {file_path}")
    with open(file_path, 'r') as file:
        key = file.read().strip()
    if not key:
        raise ValueError("API key file is empty or contains only whitespace")
    return key

client = OpenAI(api_key=load_api_key())

# ============================================================
# SAFE CHECKPOINT LOAD
# ============================================================

add_safe_globals([LabelEncoder])

checkpoint_path = os.path.join(config.OUTPUT_DIR, backbone_model)
if not os.path.exists(checkpoint_path):
    raise FileNotFoundError(f"Checkpoint not found: {checkpoint_path}")

checkpoint = torch.load(checkpoint_path, map_location=DEVICE, weights_only=False)

concept_info   = checkpoint["concept_info"]
label_encoders = checkpoint["label_encoders"]
backbone_name  = config.BACKBONE
print("Checkpoint loaded successfully.")

# ============================================================
# REBUILD MODEL
# ============================================================
model = ConceptBottleneckModel(backbone_name, concept_info).to(DEVICE)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()
print("Model restored successfully.")

# ============================================================
# REBUILD TEST LOADER
# ============================================================
meta_path = os.path.join("meta", config.DATASET_NAME)
if not os.path.exists(meta_path):
    raise FileNotFoundError(f"Metadata file not found: {meta_path}")

meta = pd.read_csv(meta_path)
meta["diagnosis"] = meta["diagnosis"].astype(str).str.lower()
meta["diagnosis"] = meta["diagnosis"].apply(lambda x: 1 if "melanoma" in x else 0)

test_idx   = meta[meta['split'] == 'test'].index.values
test_df    = meta.iloc[test_idx].copy()

valid_dataset = DermDataset(
    test_df,
    config.IMAGES_DIR,
    concept_info=concept_info,
    label_encoders=label_encoders,
    transform=get_val_transforms()
)

valid_loader = DataLoader(valid_dataset, batch_size=8)
print("Test dataset rebuilt successfully.")

# ============================================================
# CONCEPT STATE PREVALENCE (computed from dataset)
# ============================================================
# For each class (melanoma=1, nevus=0), compute the prevalence of every
# concept state (e.g. reticular_lines=absent) as a fraction in [0, 1].
# This is used by the text explanation to report data-grounded prevalence
# instead of relying on hard-coded clinical ontologies.

def compute_concept_state_prevalence(meta_df, concept_columns, label_encoders, target_column="diagnosis"):
    """
    Compute, for every class and every concept state, the fraction of
    samples in that class that exhibit the state.

    Returns
    -------
    prevalence : dict
        prevalence[class_idx]["concept_name=state"] = float in [0, 1]
    """
    prevalence = {0: {}, 1: {}}
    for cls in [0, 1]:
        subset = meta_df[meta_df[target_column] == cls]
        n = len(subset)
        if n == 0:
            continue
        for concept in concept_columns:
            counts = subset[concept].astype(str).value_counts()
            for state, cnt in counts.items():
                key = f"{concept}={state}"
                prevalence[cls][key] = round(cnt / n, 4)
    return prevalence


# Use the full meta (train + test) for stable prevalence estimates
_prev_meta = pd.read_csv(meta_path)
_prev_meta["diagnosis"] = _prev_meta["diagnosis"].astype(str).str.lower()
_prev_meta["diagnosis"] = _prev_meta["diagnosis"].apply(lambda x: 1 if "melanoma" in x else 0)

CONCEPT_STATE_PREVALENCE = compute_concept_state_prevalence(
    _prev_meta, config.CONCEPT_COLUMNS, label_encoders, target_column="diagnosis"
)
print("Concept state prevalence computed.")


# ============================================================
# CONCEPT EXPLAINER WITH SOFI
# ============================================================

class ConceptExplainer:

    def __init__(self, model, valid_loader, concept_info, label_encoders, device):
        self.model = model
        self.valid_loader = valid_loader
        self.concept_info = concept_info
        self.label_encoders = label_encoders
        self.device = device
        self.model.eval()
        self.concept_names = list(concept_info.keys())

        if not self.concept_names:
            raise ValueError("No concepts found in concept_info")

        self.head_slices = self._compute_head_slices()
        self.classifier_cache = OrderedDict()
        self.cache_maxsize = 4096
        self.concept_to_idx = {name: i for i, name in enumerate(self.concept_names)}

    def _compute_head_slices(self):
        slices = {}
        start = 0
        for c in self.concept_names:
            dim = self.concept_info[c]["n_classes"]
            slices[c] = (start, start + dim)
            start += dim
        return slices

    @torch.no_grad()
    def _cached_classifier_prob(self, concept_vector, target_class=1):
        key = concept_vector.cpu().numpy().tobytes()
        if key in self.classifier_cache:
            logits = self.classifier_cache[key]
            self.classifier_cache.move_to_end(key)
        else:
            logits = self.model.classifier(concept_vector.unsqueeze(0))
            self.classifier_cache[key] = logits
            if len(self.classifier_cache) > self.cache_maxsize:
                self.classifier_cache.popitem(last=False)
        return torch.softmax(logits, dim=1)[0, target_class].item()

    # ===================================================================
    # ZERO MARGINALIZATION
    # ===================================================================
    def _ablate_concept(self, vec, start, end):
        """Zero out the concept block — true marginalization."""
        vec[start:end] = 0.0

    # ===================================================================
    # GREEDY SCORING
    # ===================================================================
    def _scores_greedy(self, Px, target_class=1):
        """Compute initial ranking — most harmful concept first."""
        Px_dev = Px.to(self.device)
        scores = np.zeros(len(self.concept_names))
        for i, c in enumerate(self.concept_names):
            start, end = self.head_slices[c]
            perturbed = Px_dev.clone()
            self._ablate_concept(perturbed, start, end)
            scores[i] = self._cached_classifier_prob(perturbed, target_class)

        idx = np.argsort(scores)
        return [self.concept_names[i] for i in idx]

    # ===================================================================
    # CURVE BUILDING
    # ===================================================================
    def _build_curve(self, Px, ranking, target_class=1):
        """MoRF / LeRF curve using zero marginalization."""
        values = np.zeros(len(ranking) + 1)
        current = Px.to(self.device).clone()
        values[0] = self._cached_classifier_prob(current, target_class)

        for i, concept in enumerate(ranking, 1):
            start, end = self.head_slices[concept]
            self._ablate_concept(current, start, end)
            values[i] = self._cached_classifier_prob(current, target_class)
        return values

    def _compute_auc(self, curve):
        return np.trapz(curve, np.arange(len(curve)))

    def evaluate_ranking(self, Px, ranking, target_class=1):

        if not ranking:
            raise ValueError("Ranking is empty")

        morf = self._build_curve(Px, ranking, target_class)
        lerf = self._build_curve(Px, ranking[::-1], target_class)
        morf_auc = self._compute_auc(morf)
        lerf_auc = self._compute_auc(lerf)
        degradation = lerf_auc - morf_auc
        below = np.flatnonzero(morf < 0.5)
        k = below[0] if below.size > 0 else len(ranking)

        return morf, lerf, morf_auc, lerf_auc, degradation, k

    # ===================================================================
    # GREEDY + SWAP OPTIMIZATION
    # ===================================================================
    def _optimize_sofi(self, Px, target_class=1, max_iter=1000, patience=500, seed=42):
        rng = np.random.default_rng(seed)
        ranking = self._scores_greedy(Px, target_class)

        best_deg = self.evaluate_ranking(Px, ranking, target_class)[4]
        best_ranking = ranking.copy()
        no_improve = 0

        for _ in range(max_iter):
            i, j = rng.choice(len(ranking), 2, replace=False)
            candidate = best_ranking.copy()
            candidate[i], candidate[j] = candidate[j], candidate[i]
            candidate_deg = self.evaluate_ranking(Px, candidate, target_class)[4]

            if candidate_deg > best_deg:
                best_ranking = candidate
                best_deg = candidate_deg
                no_improve = 0
            else:
                no_improve += 1
            if no_improve >= patience:
                break
        return best_ranking

    def optimize(self, Px, target_class=1, max_iter=1000, patience=500, seed=42):
        """Main entry point — returns SOFI solution with zero marginalization."""
        ranking = self._optimize_sofi(Px, target_class, max_iter, patience, seed)
        morf, lerf, morf_auc, lerf_auc, degradation, k = self.evaluate_ranking(Px, ranking, target_class)
        return {
            "ranking":     ranking,
            "topk":        ranking[:k],
            "k":           k,
            "morf_curve":  morf,
            "lerf_curve":  lerf,
            "morf_auc":    morf_auc,
            "lerf_auc":    lerf_auc,
            "degradation": degradation,
        }


# ============================================================
# TEXT EXPLANATION
# ============================================================

def generate_text_explanation(
    solution,
    concept_logits_raw,
    logits,
    concept_info,
    label_encoders,
    concept_state_prevalence,
    gpt_model="gpt-4o"
):
    probs = torch.softmax(logits, dim=1)
    melanoma_prob = probs[0, 1].item()
    nevus_prob    = probs[0, 0].item()

    target_class = 1 if melanoma_prob > 0.5 else 0
    class_name   = "melanoma" if target_class == 1 else "nevus"
    class_prob   = melanoma_prob if target_class == 1 else nevus_prob

    k = solution["k"]
    concept_names = list(concept_info.keys())

    # ------------------------------------------------------------------
    # Decode all concept states and build ranking text
    # ------------------------------------------------------------------
    decoded_states    = {}
    full_ranking_text = []
    topk_text         = []

    for rank, concept_name in enumerate(solution["ranking"]):
        head_idx    = concept_names.index(concept_name)
        head_logits = concept_logits_raw[head_idx]
        head_probs  = torch.softmax(head_logits, dim=1)[0]

        pred_class_concept = torch.argmax(head_probs).item()
        pred_prob_concept  = head_probs[pred_class_concept].item()

        decoded_label = label_encoders[concept_name].inverse_transform(
            [pred_class_concept]
        )[0]

        decoded_states[concept_name] = decoded_label

        entry = f"{rank+1}. {concept_name.replace('_', ' ')}: {decoded_label} ({pred_prob_concept:.2f})"
        full_ranking_text.append(entry)
        if concept_name in solution["topk"]:
            topk_text.append(entry)

    full_ranking_str = "\n".join(full_ranking_text)
    topk_str         = "\n".join(topk_text)

    # ------------------------------------------------------------------
    # Build prevalence report for the top-k concept states
    # in the predicted class (data-driven, no ontology)
    # ------------------------------------------------------------------
    class_prevalence = concept_state_prevalence.get(target_class, {})
    prevalence_lines = []
    for rank, concept_name in enumerate(solution["topk"]):
        state = decoded_states[concept_name]
        state_key = f"{concept_name}={state}"
        prev = class_prevalence.get(state_key, 0.0)
        concept_label = concept_name.replace('_', ' ')
        prevalence_lines.append(
            f"{rank+1}. {concept_label}={state}: "
            f"prevalence in {class_name} = {prev:.2%}"
        )
    prevalence_section_str = "\n".join(prevalence_lines)

    # ------------------------------------------------------------------
    # Ancillary strings
    # ------------------------------------------------------------------
    concept_vocab = {
        name: label_encoders[name].classes_.tolist()
        for name in concept_names
    }
    concept_vocab_str = "\n".join(
        f"  {name}: {classes}" for name, classes in concept_vocab.items()
    )

    degradation     = solution["degradation"]
    reliability_msg = ""
    if degradation < 0:
        reliability_msg = "Warning: Explanation may be unreliable (negative degradation)."
    elif np.allclose(solution["morf_curve"], solution["lerf_curve"], atol=0.05):
        reliability_msg = "Warning: MoRF and LeRF curves are similar, indicating poor sparsity."

    morf_table = []
    for i, concept_name in enumerate(solution["ranking"][:k]):
        start_prob = solution["morf_curve"][i]
        end_prob   = solution["morf_curve"][i + 1]
        morf_table.append(f"{i+1}. {concept_name}: {start_prob:.2f} -> {end_prob:.2f}")
    morf_table_str = "\n".join(morf_table)

    if solution["morf_curve"][-1] < 0.5:
        drop_msg = (
            f"After progressively marginalizing the top-{k} concepts, the probability "
            f"drops below 0.5 and the model no longer recognizes the lesion as {class_name}."
        )
    else:
        drop_msg = (
            f"After progressively marginalizing the top-{k} concepts, the prediction "
            f"confidence for {class_name} substantially decreases."
        )

    # ------------------------------------------------------------------
    # GPT prompt
    # ------------------------------------------------------------------
    prompt = f"""
You are an expert dermatologist reviewing a dermoscopic image analyzed by a Concept Bottleneck Model.
Output EXACTLY three sections with the headings below. No introduction, no conclusion, no extra text. No bullet points anywhere.
When referring to concept names, replace underscores with spaces for readability.

---

(1) Diagnosis
A concise clinical report (2–3 sentences) based solely on the top-{k} concepts. State the predicted diagnosis and which features drive it.
Do not mention probabilities, model internals, or clinical plausibility here.

(2) Rationale
Begin this section with this sentence verbatim:
"This section presents the full ranking of concepts ordered from most to least contributory to the prediction, along with the model's confidence that each feature is present in the lesion."

Then output the full ranked concept list as two labeled groups:

Primary concepts (top-{k}, most contributory):
{topk_str}

Secondary concepts (remaining, least contributory):
{full_ranking_str[len(topk_str):].strip()}

Then include this paragraph verbatim:
"The MoRF curve ablates the most contributory concepts first. {drop_msg} The LeRF curve ablates the least contributory concepts first, so slower degradation is expected. The gap between MoRF and LeRF quantifies explanation quality (degradation = {degradation:.3f})."

(3) Plausibility Assessment
The following table reports the prevalence of each top-{k} primary concept state in the {class_name} class, as observed in the dataset. Report this information exactly as given — do not add, modify, or invent prevalence figures.

{prevalence_section_str}

Now write a clinical narrative discussing every top-{k} primary concept in terms of theire prevalence — no concept may be skipped.
Concepts with prevalence scores below 50% must analyzed with caution. Prevalence score below 5% must be seen as potential contradictory of the evidence.

Do not repeat the concept list. Do not add headings. Write continuous prose.
Never introduce or discuss any secondary concept.

---

INPUT DATA:
Predicted class: {class_name} (confidence: {class_prob:.2f})

Concept vocabulary (all possible values per concept):
{concept_vocab_str}

Full concept ranking:
{full_ranking_str}

Top-{k} concepts:
{topk_str}

MoRF ablation steps (top-{k}):
{morf_table_str}

{reliability_msg}
"""

    response = client.chat.completions.create(
        model=gpt_model,
        temperature=0.0,
        messages=[
            {
                "role": "system",
                "content": (
                    "You are a strict, factual dermatologist. "
                    "Output ONLY the three headings and their content — nothing else. "
                    "No introductions, no conclusions, no extra text outside the three sections. "
                    "In section (3): reproduce the prevalence table exactly as given, then "
                    "write the clinical narrative discussing every top-k concept — each as either "
                    "supportive or uninformative. Never skip a concept. "
                    "Never discuss secondary concepts."
                )
            },
            {"role": "user", "content": prompt},
        ],
    )

    return response.choices[0].message.content.strip()


# ============================================================
# PLOTTING
# ============================================================
def plot_solution(sol, img_tensor):
    fig, axes = plt.subplots(1, 2, figsize=(10, 5))
    img = img_tensor.permute(1, 2, 0).cpu().numpy()
    img = (img - img.min()) / (img.max() - img.min() + 1e-8)

    axes[0].imshow(img)
    axes[0].set_title("Image")
    x = np.arange(len(sol["morf_curve"]))

    axes[1].plot(sol["morf_curve"], marker='s', label="MoRF", markersize=8, color='tab:blue')
    axes[1].plot(sol["lerf_curve"], marker='s', label="LeRF", markersize=8, color='#8c564b')
    axes[1].fill_between(x, sol["morf_curve"], sol["lerf_curve"], color='grey', alpha=0.3)
    axes[1].axhline(0.5, linestyle='--', color='black')
    axes[1].set_title("Curves")
    axes[1].set_xlabel("Concepts Removed")
    axes[1].set_ylabel("Probability")
    axes[1].legend()
    axes[1].grid(True)

    plt.tight_layout()
    # plt.savefig("explanation.pdf", bbox_inches="tight", dpi=300)
    plt.show()


Checkpoint loaded successfully.
Froze first 0 out of 9 layers in backbone (0%)
Model restored successfully.
Test dataset rebuilt successfully.
Concept state prevalence computed.


# Web interface

In [14]:
import gradio as gr
from PIL import Image, ImageDraw
import matplotlib.pyplot as plt

import numpy as np
import torch
import os
import base64

import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

sns.set_context("notebook", font_scale=0.9)

val_transform = get_val_transforms()

# Initialize explainer once
explainer = ConceptExplainer(
    model=model,
    valid_loader=valid_loader,
    concept_info=concept_info,
    label_encoders=label_encoders,
    device=DEVICE
)
print("Explainer ready")

GPT_MODELS = ["gpt-5.4", "gpt-5.2", "gpt-4o"]

FAVICON_PATH = os.path.abspath("favicon.png")

def _make_favicon():
    size = 64
    img = Image.new('RGBA', (size, size), (0, 0, 0, 0))
    draw = ImageDraw.Draw(img)
    draw.ellipse([0, 0, 63, 63], fill=(31, 119, 180, 255))
    draw.ellipse([14, 16, 50, 48], fill=(13, 63, 110, 220))
    draw.ellipse([20, 22, 44, 44], fill=(8, 33, 58, 245))
    nodes = [(16,22),(48,22),(14,38),(50,38),(32,10),(32,54)]
    for x, y in nodes:
        draw.ellipse([x-3, y-3, x+3, y+3], fill=(126, 200, 247, 255))
    connections = [
        (nodes[0], nodes[4]), (nodes[1], nodes[4]),
        (nodes[2], nodes[5]), (nodes[3], nodes[5]),
        (nodes[0], nodes[2]), (nodes[1], nodes[3]),
    ]
    for (x1, y1), (x2, y2) in connections:
        draw.line([x1, y1, x2, y2], fill=(126, 200, 247, 130), width=1)
    draw.rectangle([44, 7, 47, 16], fill=(255, 255, 255, 230))
    draw.rectangle([41, 10, 50, 13], fill=(255, 255, 255, 230))
    img.save(FAVICON_PATH)
    print(f"Favicon saved to: {FAVICON_PATH}")
    print(f"Favicon exists:   {os.path.exists(FAVICON_PATH)}")

_make_favicon()

def _get_favicon_b64():
    with open(FAVICON_PATH, "rb") as f:
        return base64.b64encode(f.read()).decode("utf-8")

def preprocess_image(image: Image.Image):
    if image is None:
        return None
    image = image.convert("RGB")
    img_tensor = val_transform(image)
    return img_tensor

def predict_and_explain(image, max_iter, patience_pct, gpt_model, explainer_state):
    global explainer

    if image is None:
        return None, "Please upload a dermoscopic image.", explainer

    model.eval()

    img_tensor = preprocess_image(image)
    if img_tensor is None:
        return None, "Invalid image.", explainer

    img_tensor = img_tensor.unsqueeze(0).to(DEVICE)

    with torch.no_grad():
        concept_logits, logits = model(img_tensor)

    probs      = torch.softmax(logits, dim=1)
    pred_class = torch.argmax(probs, dim=1).item()
    pred_prob  = probs[0, pred_class].item()
    class_name = "melanoma" if pred_class == 1 else "nevus"

    patience = max(1, int(max_iter * patience_pct / 100))

    concept_names = list(concept_info.keys())
    Px_list = []
    for j, cp in enumerate(concept_logits):
        logits_head        = cp.squeeze(0)
        pred_class_concept = torch.argmax(logits_head).item()
        one_hot            = torch.zeros(
            concept_info[concept_names[j]]["n_classes"],
            dtype=torch.float32
        )
        one_hot[pred_class_concept] = 1.0
        Px_list.append(one_hot)

    Px = torch.cat(Px_list, dim=0)

    try:
        solution = explainer.optimize(
            Px, target_class=pred_class,
            max_iter=max_iter,
            patience=patience,
            seed=0
        )
    except Exception as e:
        return None, f"Explanation failed: {str(e)}", explainer

    text_explanation = generate_text_explanation(
        solution,
        concept_logits,
        logits,
        concept_info,
        label_encoders,
        concept_state_prevalence=CONCEPT_STATE_PREVALENCE,
        gpt_model=gpt_model
    )

    # Create figure and axes with seaborn style
    fig, ax = plt.subplots(figsize=(5.5, 4.2))

    x = np.arange(len(solution["morf_curve"]))

    # Use seaborn lineplot for better integration
    sns.lineplot(x=x, y=solution["morf_curve"], marker='s', label="MoRF", 
                markersize=10, color='tab:blue', ax=ax)
    sns.lineplot(x=x, y=solution["lerf_curve"], marker='s', label="LeRF", 
                markersize=10, color='#8c564b', ax=ax)

    # Fill between (still using matplotlib as seaborn doesn't have direct fill_between)
    ax.fill_between(x, solution["morf_curve"], solution["lerf_curve"], 
                    color='grey', alpha=0.25)
    ax.axhline(0.5, color='black', linestyle='--')

    # Use seaborn styling for labels
    ax.set_xlabel("Concepts Removed")
    ax.set_ylabel("Probability")
    # ax.set_title(f"Predicted: {class_name.capitalize()} ({pred_prob:.2f})")

    # Remove grid (seaborn might add it by default)
    ax.grid(False)

    # Set grey outer borders (spines)
    for spine in ax.spines.values():
        spine.set_color('grey')
        spine.set_linewidth(1)

    # Save and close
    # plt.savefig("explanation.pdf", bbox_inches="tight", dpi=300)
    plt.tight_layout(pad=0.8)
    plt.close(fig)

    return fig, text_explanation, explainer


theme = gr.themes.Default(
    primary_hue=gr.themes.colors.blue,
    secondary_hue=gr.themes.colors.blue,
)

custom_css = """
#submit-btn {
    background-color: #1f77b4 !important;
    color: white !important;
}

/* Hide native radio */
input[type=radio] {
    appearance: none !important;
    -webkit-appearance: none !important;
    width: 16px !important;
    height: 16px !important;
    border: 2px solid #aaa !important;
    border-radius: 50% !important;
    background: white !important;
    cursor: pointer !important;
    position: relative !important;
    margin-right: 4px !important;
    vertical-align: middle !important;
    flex-shrink: 0 !important;
}

/* Blue dot when selected */
input[type=radio]:checked {
    border-color: #1f77b4 !important;
    background: white !important;
}

input[type=radio]:checked::after {
    content: "" !important;
    display: block !important;
    width: 8px !important;
    height: 8px !important;
    border-radius: 50% !important;
    background: #1f77b4 !important;
    position: absolute !important;
    top: 50% !important;
    left: 50% !important;
    transform: translate(-50%, -50%) !important;
}
"""

favicon_b64 = _get_favicon_b64()
head_html = f'<link rel="icon" type="image/png" href="data:image/png;base64,{favicon_b64}">'

with gr.Blocks(css=custom_css, theme=theme, title="Dr. SOFI", head=head_html) as demo:

    gr.Markdown("## Concept Bottleneck Model — Dr. SOFI")

    explainer_state = gr.State(None)

    with gr.Row():

        with gr.Column(scale=1):
            image_input = gr.Image(type="pil", label="Upload Dermoscopic Image")

            with gr.Accordion("SOFI Settings", open=True):
                max_iter_input = gr.Radio(
                    choices=[100, 200, 500, 1000],
                    value=200,
                    label="Max Iterations",
                    info="Number of swap attempts in the SOFI optimizer."
                )
                patience_input = gr.Radio(
                    choices=[10, 20, 50],
                    value=10,
                    label="Patience (% of max iterations)",
                    info="Early stopping threshold as a percentage of max iterations."
                )

            with gr.Accordion("LLM Settings", open=True):
                gpt_model_input = gr.Dropdown(
                    choices=GPT_MODELS,
                    value="gpt-4o",
                    label="GPT Model",
                    info="Model used to generate the clinical text explanation."
                )

            submit_btn = gr.Button("Submit", elem_id="submit-btn")

        with gr.Column(scale=1):
            plot_output = gr.Plot(label="MoRF / LeRF Curves")
            text_output = gr.Textbox(label="Explanation", lines=15)

    submit_btn.click(
        fn=predict_and_explain,
        inputs=[image_input, max_iter_input, patience_input, gpt_model_input, explainer_state],
        outputs=[plot_output, text_output, explainer_state],
    )

demo.queue(max_size=12).launch(
    share=False,
    inbrowser=True,
    debug=False
)

Explainer ready
Favicon saved to: c:\Users\napolesr\OneDrive - Tilburg University\Desktop\demo\favicon.png
Favicon exists:   True
* Running on local URL:  http://127.0.0.1:7865
* To create a public link, set `share=True` in `launch()`.
